# ViT layer-metrics comparison

Loads `layer_metrics_*.csv` files produced by `run_vit_experiments.py` and the executed papermill notebooks (for accuracy), then plots algorithm comparisons in the same shape as the SmolLM viz.

**Adapted from the SmolLM viz with two changes:**

1. ViT CSVs have `layer_name` like `blocks.7.attn.qkv` (not `layers.7.q_proj`) and **no separate** `layer_idx` / `layer_type` columns. We parse them out of `layer_name` at load time. ViT layer types are: `attn.qkv`, `attn.proj`, `mlp.fc1`, `mlp.fc2`.
2. There's an extra section pulling **end-task accuracy** (baseline / single-layer / all-layers) from the papermill scraps, since ImageNet accuracy is the headline metric for ViT — not just `rel_error`.

For each metric of interest we produce **four delta views** (vs a chosen baseline):

| | per layer type | per transformer block |
|---|---|---|
| **aggregated** | bar chart, 1 bar per (layer type × alg), averaged across blocks | line plot, x=block idx, 1 line per alg, averaged across layer types |
| **not aggregated** | strip dots, 1 column per layer type, 1 dot per block | 1 subplot per algorithm, x=block idx, 1 line per layer type |

Time is only shown per transformer block, not aggregated.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display

# Where the per-layer CSVs live (written by ViT-pruning-and-eval.ipynb)
OUTPUT_FOLDER = "benchmark_csvs_vit"

# Where the executed papermill notebooks live (written by run_vit_experiments.py)
NOTEBOOK_FOLDER = "vit_experiment_results"

# ViT layer types (the suffix after `blocks.{idx}.`)
LAYER_TYPES = ["attn.qkv", "attn.proj", "mlp.fc1", "mlp.fc2"]
LAYER_GROUP = {
    "attn.qkv":  "Attention",
    "attn.proj": "Attention",
    "mlp.fc1":   "MLP",
    "mlp.fc2":   "MLP",
}

# Consistent color per algorithm across all plots
ALG_COLORS = {
    "GA-TETRIS":             "#d62728",
    "Original TETRIS":       "#1f77b4",
    "Sort-by-Norm":          "#2ca02c",
    "Block-Wanda":           "#ff7f0e",
    "Block-only":            "#ff7f0e",  # alias
    "Random":                "#9467bd",
    "Random-Swaps":          "#8c564b",
    "Random-Swaps (sorted)": "#17becf",
    "No prune":              "#7f7f7f",
}
def alg_color(alg):
    return ALG_COLORS.get(alg, "#555555")

# Consistent color per layer type
LAYER_COLORS = {
    "attn.qkv":  "#1f77b4",
    "attn.proj": "#d62728",
    "mlp.fc1":   "#9467bd",
    "mlp.fc2":   "#8c564b",
}
def layer_color(lt):
    return LAYER_COLORS.get(lt, "#555555")


## 1. Load all CSVs

Parses `blocks.{idx}.{layer_type}` out of `layer_name` so the rest of the notebook can use `layer_idx` and `layer_type` like the SmolLM viz does.

In [ ]:
_LAYER_RE = re.compile(r"blocks\.(\d+)\.(.+)")

def parse_layer_name(name):
    """Parse 'blocks.7.attn.qkv' -> (7, 'attn.qkv'). Returns (None, name) if it doesn't match."""
    m = _LAYER_RE.match(name)
    if not m:
        return None, name
    return int(m.group(1)), m.group(2)


csv_files = sorted(glob.glob(os.path.join(OUTPUT_FOLDER, "layer_metrics_*.csv")))
print(f"Found {len(csv_files)} CSVs:")
for f in csv_files:
    print(f"  {os.path.basename(f)}")

dfs = [pd.read_csv(f) for f in csv_files]
df = pd.concat(dfs, ignore_index=True)

# Parse layer_name -> layer_idx + layer_type
parsed = df["layer_name"].apply(parse_layer_name)
df["layer_idx"]  = parsed.apply(lambda x: x[0])
df["layer_type"] = parsed.apply(lambda x: x[1])

# Drop layers we couldn't parse (warn loudly)
unparsed = df[df["layer_idx"].isna()]
if len(unparsed) > 0:
    print(f"\nWARNING: {len(unparsed)} rows had layer_name not matching 'blocks.N.SUFFIX':")
    print(unparsed["layer_name"].unique())
df = df[df["layer_idx"].notna()].copy()
df["layer_idx"] = df["layer_idx"].astype(int)

# Derived columns (same as SmolLM viz)
df["retained_frac"]     = df["retained_wanda_mass"] / df["total_wanda_mass"]
df["pruned_wanda_mass"] = df["total_wanda_mass"] - df["retained_wanda_mass"]
df["rel_error_weights"] = df["pruned_weight_l1"] / df["total_weight_l1"]
df["block_size"]        = df["block_rows"].astype(str) + "x" + df["block_cols"].astype(str)
df["layer_group"]       = df["layer_type"].map(LAYER_GROUP)

# If `sort_start` exists in the CSV (it does for ViT), promote to a per-experiment
# discriminator so SORT_START=True/False random_swaps don't collapse together.
if "sort_start" in df.columns:
    is_random_swaps = df["algorithm"].isin(["random_swaps", "random_swaps_find_mask"])
    df.loc[is_random_swaps & (df["sort_start"] == True), "algorithm"] = (
        df.loc[is_random_swaps & (df["sort_start"] == True), "algorithm"] + "_sort_start"
    )

# Map algorithm codenames to thesis display names
ALG_DISPLAY = {
    "our_tetris":                       "GA-TETRIS",
    "original_tetris":                  "Original TETRIS",
    "block_wanda":                      "Block-Wanda",
    "block_only":                       "Block-only",
    "sort_columns_by_norm":             "Sort-by-Norm",
    "random_permutation_pruning":       "Random",
    "random_swaps":                     "Random-Swaps",
    "random_swaps_find_mask":           "Random-Swaps",
    "random_swaps_sort_start":          "Random-Swaps (sorted)",
    "random_swaps_find_mask_sort_start":"Random-Swaps (sorted)",
    "no_prune":                         "No prune",
}
df["algorithm"] = df["algorithm"].map(ALG_DISPLAY).fillna(df["algorithm"])

print(f"\nTotal rows: {len(df)}")
print(f"Algorithms:  {sorted(df.algorithm.unique())}")
print(f"Layer types: {sorted(df.layer_type.unique())}")
print(f"Block idxs:  {sorted(df.layer_idx.unique())}")
print(f"Block sizes: {sorted(df.block_size.unique())}")
print(f"Sparsities:  {sorted(df.sparsity.unique())}")


## 2. Pick an experiment config to compare

Fix everything except `algorithm` to compare algorithms head-to-head.

In [ ]:
# Filter knobs — edit to pick which experiment slice to compare
FILTER_BLOCK_ROWS = 1
FILTER_BLOCK_COLS = 2
FILTER_SPARSITY   = 0.5

mask = (
    (df.block_rows == FILTER_BLOCK_ROWS) &
    (df.block_cols == FILTER_BLOCK_COLS) &
    (df.sparsity   == FILTER_SPARSITY)
)
df_exp = df[mask].copy()
print(f"Filtered rows: {len(df_exp)}")
print(f"Algorithms in filter: {sorted(df_exp.algorithm.unique())}")

# Baseline to compute deltas against
BASELINE_ALG = "Block-Wanda"
assert BASELINE_ALG in df_exp.algorithm.unique(), \
    f"Baseline '{BASELINE_ALG}' not found. Available: {sorted(df_exp.algorithm.unique())}"


## 3. End-task accuracy

Accuracy is in the CSV (one value per algorithm, repeated across layers).
Pull the unique values per (algorithm, block_size, sparsity) — they're constant within a single experiment.

In [ ]:
# Accuracy is now in the CSV (one value per algorithm, repeated across layers).
# Pull the unique values per (algorithm, block_size, sparsity) — they're constant
# within a single experiment.

acc_cols = ["baseline_accuracy", "accuracy_single_layer",
            "accuracy_all_layers", "single_layer_name"]

if all(c in df.columns for c in acc_cols):
    acc_df = (df.groupby(["algorithm", "block_rows", "block_cols", "sparsity"])
                .agg({c: "first" for c in acc_cols})
                .reset_index())
    print(f"Found accuracy data for {len(acc_df)} experiments.")
    display(acc_df.round(4))
else:
    missing = [c for c in acc_cols if c not in df.columns]
    print(f"Accuracy columns missing from CSVs: {missing}")
    print("CSVs were generated before accuracy was added — re-run experiments to populate.")
    acc_df = None

## 4. Helper functions

Same four-views machinery as the SmolLM viz. The shape of the data here (after parsing `layer_name`) matches the SmolLM CSVs exactly, so these helpers work without modification.

In [ ]:
def compute_delta(data, metric, baseline_alg):
    """Per-layer delta vs baseline. Returns long-format frame with delta column."""
    piv = data.pivot_table(
        index=["layer_type", "layer_idx"],
        columns="algorithm",
        values=metric,
        aggfunc="mean",
    )
    delta = piv.subtract(piv[baseline_alg], axis=0).drop(columns=[baseline_alg])
    return (delta.reset_index()
                 .melt(id_vars=["layer_type", "layer_idx"],
                       var_name="algorithm", value_name="delta")
                 .dropna(subset=["delta"]))


# View 1/4: per layer type, aggregated  (bar chart)
def delta_layertype_aggregated(data, metric, baseline_alg, ylabel, title_suffix=""):
    long = compute_delta(data, metric, baseline_alg)
    agg = (long.groupby(["layer_type", "algorithm"])["delta"]
              .mean().unstack("algorithm")
              .reindex([lt for lt in LAYER_TYPES if lt in long.layer_type.unique()]))

    algs = list(agg.columns)
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(agg.index))
    width = 0.8 / max(len(algs), 1)

    for k, alg in enumerate(algs):
        ax.bar(x + k * width, agg[alg].values, width,
               label=alg, color=alg_color(alg), alpha=0.9)

    ax.axhline(0, color="black", linewidth=1.0)
    ax.set_xticks(x + width * (len(algs) - 1) / 2)
    ax.set_xticklabels(agg.index)
    ax.set_ylabel(f"Mean Δ {ylabel} vs {baseline_alg}\n(negative = better)")
    ax.set_title(f"Mean Δ {metric} per layer type, averaged across blocks{title_suffix}",
                 fontsize=13, fontweight="bold")
    ax.grid(True, axis="y", linestyle="--", alpha=0.35)
    ax.legend(loc="best", frameon=False)
    fig.tight_layout()
    return fig


# View 2/4: per layer type, not aggregated  (strip dots)
def delta_layertype_strip(data, metric, baseline_alg, ylabel, title_suffix=""):
    long = compute_delta(data, metric, baseline_alg)
    algs = sorted(long.algorithm.unique())
    layer_types_present = [lt for lt in LAYER_TYPES if lt in long.layer_type.unique()]

    fig, ax = plt.subplots(figsize=(14, 6))
    rng_j = np.random.default_rng(0)

    n_alg = max(len(algs), 1)
    sub_step = 0.8 / n_alg
    sub_offsets = -0.4 + sub_step / 2 + np.arange(n_alg) * sub_step

    for col_i, lt in enumerate(layer_types_present):
        for a_i, a in enumerate(algs):
            sub = long[(long.layer_type == lt) & (long.algorithm == a)]
            if len(sub) == 0:
                continue
            x_center = col_i + sub_offsets[a_i]
            jitter = rng_j.uniform(-sub_step * 0.35, sub_step * 0.35, size=len(sub))
            ax.scatter(x_center + jitter, sub["delta"].values,
                       color=alg_color(a), alpha=0.65, s=24,
                       edgecolor="white", linewidth=0.4,
                       label=a if col_i == 0 else None)
            ax.hlines(sub["delta"].mean(),
                      x_center - sub_step * 0.4, x_center + sub_step * 0.4,
                      color=alg_color(a), linewidth=2.2, zorder=3)

    ax.axhline(0, color="black", linewidth=1.1)
    ax.set_xticks(np.arange(len(layer_types_present)))
    ax.set_xticklabels(layer_types_present)
    ax.set_ylabel(f"Δ {ylabel} vs {baseline_alg}\n(negative = better)")
    ax.set_title(f"Per-block Δ {metric} grouped by layer type{title_suffix}",
                 fontsize=13, fontweight="bold")
    ax.grid(True, axis="y", linestyle="--", alpha=0.35)
    ax.legend(loc="best", frameon=False, fontsize=9)
    fig.tight_layout()
    return fig


# View 3/4: per transformer block, aggregated  (line plot)
def delta_block_aggregated(data, metric, baseline_alg, ylabel, title_suffix=""):
    long = compute_delta(data, metric, baseline_alg)
    algs = sorted(long.algorithm.unique())

    agg = (long.groupby(["layer_idx", "algorithm"])["delta"]
              .mean().unstack("algorithm"))

    fig, ax = plt.subplots(figsize=(12, 5))
    for a in algs:
        if a not in agg.columns: continue
        ax.plot(agg.index, agg[a], label=a, color=alg_color(a),
                linewidth=1.6, marker="o", markersize=4, alpha=0.9)
    ax.axhline(0, color='black', linewidth=1.2, label=f"{baseline_alg} (baseline)")
    ax.set_xlabel("Transformer block index")
    ax.set_ylabel(f"Mean Δ {ylabel} across layer types\n(negative = better)")
    ax.set_title(f"Δ {metric} by transformer block, averaged across layer types{title_suffix}",
                 fontsize=13, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend(loc="best", frameon=False)
    fig.tight_layout()
    return fig


# View 4/4: per transformer block, not aggregated  (one subplot per algorithm)
def delta_block_per_algorithm(data, metric, baseline_alg, ylabel, title_suffix=""):
    long = compute_delta(data, metric, baseline_alg)
    algs = sorted(long.algorithm.unique())
    if len(algs) == 0:
        print("No non-baseline algorithms in data.")
        return None

    n_total = len(algs) + 1  # reserve a slot for the legend
    n_cols = min(3, n_total)
    n_rows = (n_total + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(6 * n_cols, 4 * n_rows),
                             sharex=True, sharey=True, squeeze=False)
    axes_flat = axes.flatten()

    for k, a in enumerate(algs):
        ax = axes_flat[k]
        sub = long[long.algorithm == a]
        for lt in LAYER_TYPES:
            line = sub[sub.layer_type == lt].sort_values("layer_idx")
            if len(line) == 0:
                continue
            ax.plot(line["layer_idx"], line["delta"],
                    label=lt, color=layer_color(lt),
                    linewidth=1.4, marker="o", markersize=3.5, alpha=0.9)
        ax.axhline(0, color="black", linewidth=1.0)
        ax.set_title(a, fontsize=12, fontweight="bold", color=alg_color(a))
        ax.grid(True, linestyle="--", alpha=0.35)
        ax.set_xlabel("Transformer block index")
        ax.set_ylabel(f"Δ {ylabel}")

    for k in range(len(algs), len(axes_flat)):
        axes_flat[k].axis("off")

    handles = [Line2D([0], [0], color=layer_color(lt), lw=1.8,
                      marker="o", markersize=4, label=lt)
               for lt in LAYER_TYPES if lt in long.layer_type.unique()]
    axes_flat[len(algs)].legend(handles=handles, loc="center", fontsize=11,
                                frameon=False, title="Layer type", title_fontsize=12)

    fig.suptitle(f"Δ {metric} by block, per algorithm{title_suffix}",
                 fontsize=14, fontweight="bold", y=1.00)
    fig.tight_layout()
    return fig


# Strip plot: every layer as a dot, one column per algorithm
def strip_delta(data, metric, baseline_alg, ylabel, title_suffix=""):
    long = compute_delta(data, metric, baseline_alg)
    algs = sorted(long.algorithm.unique())

    fig, ax = plt.subplots(figsize=(10, 5))
    rng_j = np.random.default_rng(0)
    for k, a in enumerate(algs):
        vals = long[long.algorithm == a]["delta"].values
        if len(vals) == 0:
            continue
        jitter = rng_j.uniform(-0.25, 0.25, size=len(vals))
        ax.scatter(k + jitter, vals, color=alg_color(a), alpha=0.55,
                   s=22, edgecolor="white", linewidth=0.4)
        ax.hlines(vals.mean(), k - 0.35, k + 0.35,
                  color=alg_color(a), linewidth=2.5, zorder=3)

    ax.axhline(0, color='black', linewidth=1.1)
    ax.set_xticks(range(len(algs)))
    ax.set_xticklabels(algs, rotation=15, ha='right')
    ax.set_ylabel(f"Δ {ylabel} vs {baseline_alg}\n(negative = better)")
    ax.set_title(f"Per-layer Δ {metric}, each dot = one layer{title_suffix}",
                 fontsize=13, fontweight="bold")
    ax.grid(True, axis="y", linestyle="--", alpha=0.35)
    fig.tight_layout()
    return fig


# Time view: per transformer block, not aggregated, no delta (absolute time)
def time_block_per_algorithm(data, title_suffix=""):
    algs = sorted(data.algorithm.unique())
    n_total = len(algs) + 1
    n_cols = min(3, n_total)
    n_rows = (n_total + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(6 * n_cols, 4 * n_rows),
                             sharex=True, sharey=True, squeeze=False)
    axes_flat = axes.flatten()

    for k, a in enumerate(algs):
        ax = axes_flat[k]
        sub = data[data.algorithm == a]
        for lt in LAYER_TYPES:
            line = sub[sub.layer_type == lt].sort_values("layer_idx")
            if len(line) == 0:
                continue
            ax.plot(line["layer_idx"], line["total_time_sec"],
                    label=lt, color=layer_color(lt),
                    linewidth=1.4, marker="o", markersize=3.5, alpha=0.9)
        ax.set_title(a, fontsize=12, fontweight="bold", color=alg_color(a))
        ax.set_yscale("log")
        ax.grid(True, linestyle="--", alpha=0.35, which="both")
        ax.set_xlabel("Transformer block index")
        ax.set_ylabel("Pruning time (s, log scale)")

    for k in range(len(algs), len(axes_flat)):
        axes_flat[k].axis("off")

    handles = [Line2D([0], [0], color=layer_color(lt), lw=1.8,
                      marker="o", markersize=4, label=lt)
               for lt in LAYER_TYPES if lt in data.layer_type.unique()]
    axes_flat[len(algs)].legend(handles=handles, loc="center", fontsize=11,
                                frameon=False, title="Layer type", title_fontsize=12)

    fig.suptitle(f"Pruning time per block, per algorithm{title_suffix}",
                 fontsize=14, fontweight="bold", y=1.00)
    fig.tight_layout()
    return fig


## 5. Summary tables

In [ ]:
overall = (df_exp.groupby("algorithm")
           .agg(mean_rel_error=("rel_error", "mean"),
                mean_score_improvement_pct=("score_improvement_pct", "mean"),
                median_score_improvement_pct=("score_improvement_pct", "median"),
                mean_retained_frac=("retained_frac", "mean"),
                mean_time_sec=("total_time_sec", "mean"),
                total_time_sec=("total_time_sec", "sum"),
                n_layers=("rel_error", "count"))
           .sort_values("mean_score_improvement_pct", ascending=False))
print("=== Overall (all layers pooled) ===")
display(overall.round(5))

In [ ]:
# Per-algorithm × per-layer-type: mean rel_error
by_type = (df_exp.groupby(["layer_type", "algorithm"])["rel_error"]
           .mean().unstack("algorithm"))
by_type = by_type.reindex([lt for lt in LAYER_TYPES if lt in by_type.index])
print("=== Mean rel_error per layer type × algorithm (lower is better) ===")
display(by_type.round(5))

# Highlight best algorithm per layer type
def _highlight_min(row):
    is_min = row == row.min()
    return ["font-weight: bold; background-color: #d4f4dd" if v else "" for v in is_min]
display(by_type.round(5).style.apply(_highlight_min, axis=1))


## 6. Relative error (Wanda-weighted): four delta views

`rel_error = ‖ΔW · ‖x‖‖² / ‖W · ‖x‖‖²` — fraction of Wanda mass thrown away.

In [ ]:
SUFFIX = f"  —  block {FILTER_BLOCK_ROWS}×{FILTER_BLOCK_COLS}, sparsity {FILTER_SPARSITY}  (vs {BASELINE_ALG})"

delta_layertype_aggregated(df_exp, "rel_error", BASELINE_ALG,
                           "rel_error (fraction)", title_suffix=SUFFIX); plt.show()


In [ ]:
delta_layertype_strip(df_exp, "rel_error", BASELINE_ALG,
                      "rel_error (fraction)", title_suffix=SUFFIX); plt.show()


In [ ]:
delta_block_aggregated(df_exp, "rel_error", BASELINE_ALG,
                       "rel_error (fraction)", title_suffix=SUFFIX); plt.show()


In [ ]:
delta_block_per_algorithm(df_exp, "rel_error", BASELINE_ALG,
                          "rel_error (fraction)", title_suffix=SUFFIX); plt.show()


## 6.5. Score-space improvement: four views

`score_improvement_pct = 100 · (baseline_pruned_score − algorithm_pruned_score) / baseline_pruned_score`

This is the metric your `bigger_test_simplified` reports. Higher = better. Block-Wanda is the baseline (always 0% by definition). The interesting comparison is which algorithms push above 0% and by how much.

Note: this is **not** a delta vs Original TETRIS — it's an absolute "how much better than block-only" per algorithm. So we plot raw values, not deltas.

In [ ]:
def absolute_layertype_aggregated(data, metric, ylabel, title_suffix=""):
    """Bar chart of absolute metric (no baseline subtraction)."""
    agg = (data.groupby(["layer_type", "algorithm"])[metric]
              .mean().unstack("algorithm")
              .reindex([lt for lt in LAYER_TYPES if lt in data.layer_type.unique()]))
    algs = list(agg.columns)
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(agg.index))
    width = 0.8 / max(len(algs), 1)
    for k, alg in enumerate(algs):
        ax.bar(x + k * width, agg[alg].values, width,
               label=alg, color=alg_color(alg), alpha=0.9)
    ax.axhline(0, color="black", linewidth=1.0)
    ax.set_xticks(x + width * (len(algs) - 1) / 2)
    ax.set_xticklabels(agg.index)
    ax.set_ylabel(f"Mean {ylabel}")
    ax.set_title(f"Mean {metric} per layer type, averaged across blocks{title_suffix}",
                 fontsize=13, fontweight="bold")
    ax.grid(True, axis="y", linestyle="--", alpha=0.35)
    ax.legend(loc="best", frameon=False)
    fig.tight_layout()
    return fig


def absolute_layertype_strip(data, metric, ylabel, title_suffix=""):
    """Strip dots: 1 column per layer type, 1 dot per block, color = algorithm."""
    algs = sorted(data.algorithm.unique())
    layer_types_present = [lt for lt in LAYER_TYPES if lt in data.layer_type.unique()]
    fig, ax = plt.subplots(figsize=(14, 6))
    rng_j = np.random.default_rng(0)
    n_alg = max(len(algs), 1)
    sub_step = 0.8 / n_alg
    sub_offsets = -0.4 + sub_step / 2 + np.arange(n_alg) * sub_step
    for col_i, lt in enumerate(layer_types_present):
        for a_i, a in enumerate(algs):
            sub = data[(data.layer_type == lt) & (data.algorithm == a)]
            if len(sub) == 0:
                continue
            x_center = col_i + sub_offsets[a_i]
            jitter = rng_j.uniform(-sub_step * 0.35, sub_step * 0.35, size=len(sub))
            ax.scatter(x_center + jitter, sub[metric].values,
                       color=alg_color(a), alpha=0.65, s=24,
                       edgecolor="white", linewidth=0.4,
                       label=a if col_i == 0 else None)
            ax.hlines(sub[metric].mean(),
                      x_center - sub_step * 0.4, x_center + sub_step * 0.4,
                      color=alg_color(a), linewidth=2.2, zorder=3)
    ax.axhline(0, color="black", linewidth=1.1)
    ax.set_xticks(np.arange(len(layer_types_present)))
    ax.set_xticklabels(layer_types_present)
    ax.set_ylabel(ylabel)
    ax.set_title(f"Per-block {metric} grouped by layer type{title_suffix}",
                 fontsize=13, fontweight="bold")
    ax.grid(True, axis="y", linestyle="--", alpha=0.35)
    ax.legend(loc="best", frameon=False, fontsize=9)
    fig.tight_layout()
    return fig


def absolute_block_aggregated(data, metric, ylabel, title_suffix=""):
    """Line plot: x=block idx, y=mean metric across layer types, line per algorithm."""
    algs = sorted(data.algorithm.unique())
    agg = (data.groupby(["layer_idx", "algorithm"])[metric]
              .mean().unstack("algorithm"))
    fig, ax = plt.subplots(figsize=(12, 5))
    for a in algs:
        if a not in agg.columns: continue
        ax.plot(agg.index, agg[a], label=a, color=alg_color(a),
                linewidth=1.6, marker="o", markersize=4, alpha=0.9)
    ax.axhline(0, color="black", linewidth=1.0)
    ax.set_xlabel("Transformer block index")
    ax.set_ylabel(f"Mean {ylabel} across layer types")
    ax.set_title(f"{metric} by transformer block, averaged across layer types{title_suffix}",
                 fontsize=13, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend(loc="best", frameon=False)
    fig.tight_layout()
    return fig


def absolute_block_per_algorithm(data, metric, ylabel, title_suffix=""):
    """One subplot per algorithm; x=block idx; line per layer type."""
    algs = sorted(data.algorithm.unique())
    n_total = len(algs) + 1
    n_cols = min(3, n_total)
    n_rows = (n_total + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows),
                             sharex=True, sharey=True, squeeze=False)
    axes_flat = axes.flatten()
    for k, a in enumerate(algs):
        ax = axes_flat[k]
        sub = data[data.algorithm == a]
        for lt in LAYER_TYPES:
            line = sub[sub.layer_type == lt].sort_values("layer_idx")
            if len(line) == 0:
                continue
            ax.plot(line["layer_idx"], line[metric],
                    label=lt, color=layer_color(lt),
                    linewidth=1.4, marker="o", markersize=3.5, alpha=0.9)
        ax.axhline(0, color="black", linewidth=1.0)
        ax.set_title(a, fontsize=12, fontweight="bold", color=alg_color(a))
        ax.grid(True, linestyle="--", alpha=0.35)
        ax.set_xlabel("Transformer block index")
        ax.set_ylabel(ylabel)
    for k in range(len(algs), len(axes_flat)):
        axes_flat[k].axis("off")
    handles = [Line2D([0], [0], color=layer_color(lt), lw=1.8,
                      marker="o", markersize=4, label=lt)
               for lt in LAYER_TYPES if lt in data.layer_type.unique()]
    axes_flat[len(algs)].legend(handles=handles, loc="center", fontsize=11,
                                frameon=False, title="Layer type", title_fontsize=12)
    fig.suptitle(f"{metric} by block, per algorithm{title_suffix}",
                 fontsize=14, fontweight="bold", y=1.00)
    fig.tight_layout()
    return fig

In [ ]:
absolute_layertype_aggregated(df_exp, "score_improvement_pct",
                              "score improvement (%)", title_suffix=SUFFIX); plt.show()

absolute_layertype_strip(df_exp, "score_improvement_pct",
                         "score improvement (%)", title_suffix=SUFFIX); plt.show()

absolute_block_aggregated(df_exp, "score_improvement_pct",
                          "score improvement (%)", title_suffix=SUFFIX); plt.show()

absolute_block_per_algorithm(df_exp, "score_improvement_pct",
                             "score improvement (%)", title_suffix=SUFFIX); plt.show()

## 6.6. Does score improvement translate to rel_error reduction?

X-axis: how much did this algorithm reduce the score-space pruning loss vs Block-Wanda?
Y-axis: how much did it reduce `rel_error` vs Block-Wanda? (negative = improvement)

A tight diagonal cluster means the score metric is doing its job — algorithms that find better permutations in score-space also produce smaller weight-space errors.

In [ ]:
# Compute Block-Wanda reference per (layer_type, layer_idx) for rel_error
block_wanda_alg = "Block-Wanda"
ref = (df_exp[df_exp.algorithm == block_wanda_alg]
       .set_index(["layer_type", "layer_idx"])["rel_error"])

# Build per-layer comparison vs Block-Wanda for every other algorithm
comparison_rows = []
for alg in df_exp.algorithm.unique():
    if alg == block_wanda_alg or alg == "No prune":
        continue
    sub = df_exp[df_exp.algorithm == alg].set_index(["layer_type", "layer_idx"])
    for (lt, li), row in sub.iterrows():
        if (lt, li) not in ref.index:
            continue
        comparison_rows.append({
            "algorithm": alg,
            "layer_type": lt,
            "layer_idx": li,
            "score_improvement_pct": row["score_improvement_pct"],
            "rel_error_delta": row["rel_error"] - ref[(lt, li)],
        })
comp = pd.DataFrame(comparison_rows)

# One subplot per algorithm
algs = sorted(comp.algorithm.unique())
n_total = len(algs) + 1
n_cols = min(3, n_total)
n_rows = (n_total + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows),
                         sharex=False, sharey=False, squeeze=False)
axes_flat = axes.flatten()

for k, a in enumerate(algs):
    ax = axes_flat[k]
    sub = comp[comp.algorithm == a]
    for lt in LAYER_TYPES:
        s = sub[sub.layer_type == lt]
        if len(s) == 0: continue
        ax.scatter(s["score_improvement_pct"], s["rel_error_delta"],
                   color=layer_color(lt), label=lt, s=55, alpha=0.75,
                   edgecolor="white", linewidth=0.5)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axvline(0, color="black", linewidth=0.8)
    r = sub["score_improvement_pct"].corr(-sub["rel_error_delta"])
    ax.set_title(f"{a}  (Pearson r={r:.2f})",
                 fontsize=11, fontweight="bold", color=alg_color(a))
    ax.set_xlabel("score improvement (%)")
    ax.set_ylabel("Δ rel_error vs Block-Wanda\n(negative = better)")
    ax.grid(True, linestyle="--", alpha=0.35)

for k in range(len(algs), len(axes_flat)):
    axes_flat[k].axis("off")

handles = [Line2D([0], [0], color=layer_color(lt), lw=0,
                  marker="o", markersize=8, label=lt)
           for lt in LAYER_TYPES if lt in comp.layer_type.unique()]
axes_flat[len(algs)].legend(handles=handles, loc="center", fontsize=11,
                            frameon=False, title="Layer type", title_fontsize=12)
fig.suptitle("Score improvement vs rel_error reduction (per layer)",
             fontsize=14, fontweight="bold", y=1.00)
fig.tight_layout()
plt.show()

# Summary across all algorithms pooled
print(f"Pooled correlation (score_improvement vs -rel_error_delta): "
      f"{comp['score_improvement_pct'].corr(-comp['rel_error_delta']):.3f}")
print()
print("Per-algorithm correlations:")
for a in algs:
    sub = comp[comp.algorithm == a]
    r = sub["score_improvement_pct"].corr(-sub["rel_error_delta"])
    print(f"  {a:30s}  r={r:.3f}  (n={len(sub)})")

## 7. Pruned Wanda mass (absolute): four delta views

In [ ]:
delta_layertype_aggregated(df_exp, "pruned_wanda_mass", BASELINE_ALG,
                           "pruned Wanda mass", title_suffix=SUFFIX); plt.show()

delta_layertype_strip(df_exp, "pruned_wanda_mass", BASELINE_ALG,
                      "pruned Wanda mass", title_suffix=SUFFIX); plt.show()

delta_block_aggregated(df_exp, "pruned_wanda_mass", BASELINE_ALG,
                       "pruned Wanda mass", title_suffix=SUFFIX); plt.show()

delta_block_per_algorithm(df_exp, "pruned_wanda_mass", BASELINE_ALG,
                          "pruned Wanda mass", title_suffix=SUFFIX); plt.show()



## 8. Strip-dot summary

In [ ]:
strip_delta(df_exp, "rel_error", BASELINE_ALG,
            "rel_error (fraction)", title_suffix=SUFFIX); plt.show()
strip_delta(df_exp, "pruned_wanda_mass", BASELINE_ALG,
            "pruned Wanda mass", title_suffix=SUFFIX); plt.show()


## 9. Weight-space relative error: four delta views

`rel_error_weights = ‖ΔW‖₁ / ‖W‖₁` — ignoring activations.

In [ ]:
delta_layertype_aggregated(df_exp, "rel_error_weights", BASELINE_ALG,
                           "weight-space rel_error", title_suffix=SUFFIX); plt.show()

delta_layertype_strip(df_exp, "rel_error_weights", BASELINE_ALG,
                      "weight-space rel_error", title_suffix=SUFFIX); plt.show()

delta_block_aggregated(df_exp, "rel_error_weights", BASELINE_ALG,
                       "weight-space rel_error", title_suffix=SUFFIX); plt.show()

delta_block_per_algorithm(df_exp, "rel_error_weights", BASELINE_ALG,
                          "weight-space rel_error", title_suffix=SUFFIX); plt.show()



## 10. Pruned weight L1 (absolute): four delta views

In [ ]:
delta_layertype_aggregated(df_exp, "pruned_weight_l1", BASELINE_ALG,
                           "pruned weight L1", title_suffix=SUFFIX); plt.show()

delta_layertype_strip(df_exp, "pruned_weight_l1", BASELINE_ALG,
                      "pruned weight L1", title_suffix=SUFFIX); plt.show()

delta_block_aggregated(df_exp, "pruned_weight_l1", BASELINE_ALG,
                       "pruned weight L1", title_suffix=SUFFIX); plt.show()

delta_block_per_algorithm(df_exp, "pruned_weight_l1", BASELINE_ALG,
                          "pruned weight L1", title_suffix=SUFFIX); plt.show()




## 11. Pruning time

Per block, not aggregated, log y-scale. One subplot per algorithm.

In [ ]:
time_block_per_algorithm(df_exp, title_suffix=SUFFIX); plt.show()


## 12. Compare across block sizes (single algorithm)

In [ ]:
FOCUS_ALG = ALG_DISPLAY["our_tetris"]

block_sizes = sorted(df[df.algorithm == FOCUS_ALG].block_size.unique(),
                     key=lambda s: (int(s.split("x")[0]), int(s.split("x")[1])))

if len(block_sizes) > 1:
    layer_types_present = [lt for lt in LAYER_TYPES if lt in df.layer_type.unique()]
    n_lt = len(layer_types_present)
    n_cols = min(2, n_lt)
    n_rows = (n_lt + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(8 * n_cols, 4 * n_rows),
                             sharex=True, squeeze=False)
    axes_flat = axes.flatten()
    cmap = plt.cm.viridis(np.linspace(0, 0.85, len(block_sizes)))

    for i, lt in enumerate(layer_types_present):
        ax = axes_flat[i]
        for bs, color in zip(block_sizes, cmap):
            sub = df[(df.algorithm == FOCUS_ALG) & (df.layer_type == lt) & (df.block_size == bs)]
            sub = sub.sort_values("layer_idx")
            ax.plot(sub["layer_idx"], sub["rel_error"],
                    label=f"block {bs}", color=color, linewidth=1.4, marker="o", markersize=3.5)
        ax.set_title(lt, fontsize=12, fontweight="bold")
        ax.grid(True, linestyle="--", alpha=0.35)
        if i % n_cols == 0: ax.set_ylabel("Rel. error")
        ax.set_xlabel("Transformer block index")

    for i in range(n_lt, len(axes_flat)):
        axes_flat[i].axis("off")

    handles = [Line2D([0], [0], color=c, lw=1.8, marker="o", markersize=4, label=f"block {bs}")
               for bs, c in zip(block_sizes, cmap)]
    fig.legend(handles=handles, loc="center right", frameon=False,
               title=f"{FOCUS_ALG}", bbox_to_anchor=(1.02, 0.5))
    fig.suptitle(f"Block-size sweep for {FOCUS_ALG}", fontsize=14, fontweight="bold", y=1.00)
    fig.tight_layout()
    plt.show()
else:
    print(f"Only one block size ({block_sizes[0] if block_sizes else 'none'}) in data for {FOCUS_ALG} — nothing to sweep.")


In [ ]:
# ============================================================================
# Block-width sweep — the headline figure
# ============================================================================
df_sweep = df[df.sparsity == FILTER_SPARSITY].copy()
df_sweep["block_width"] = df_sweep["block_cols"]

sweep_agg = (df_sweep.groupby(["algorithm", "block_width"])["score_improvement_pct"]
             .agg(["mean", "median", "std", "count"])
             .reset_index())

# Drop algorithms that don't have multiple block widths
algs_with_sweep = (sweep_agg.groupby("algorithm")["block_width"].nunique()
                   .pipe(lambda s: s[s > 1]).index.tolist())
sweep_agg = sweep_agg[sweep_agg["algorithm"].isin(algs_with_sweep)]

print("=== Mean score improvement (%) by algorithm × block width ===")
display(sweep_agg.pivot(index="block_width", columns="algorithm", values="mean").round(3))

fig, ax = plt.subplots(figsize=(11, 6))
for alg in sorted(algs_with_sweep):
    sub = sweep_agg[sweep_agg["algorithm"] == alg].sort_values("block_width")
    if len(sub) == 0: continue
    color = alg_color(alg)
    ax.plot(sub["block_width"], sub["mean"], marker="o", linewidth=2.0,
            markersize=8, color=color, label=alg, zorder=3)
    if "std" in sub.columns and sub["std"].notna().any():
        ax.fill_between(sub["block_width"],
                        sub["mean"] - sub["std"], sub["mean"] + sub["std"],
                        color=color, alpha=0.12, zorder=2)

ax.axhline(0, color="black", linewidth=1.0)
ax.set_xscale("log", base=2)
ax.set_xticks([2, 4, 8, 16])
ax.set_xticklabels(["1×2", "1×4", "1×8", "1×16"])
ax.set_xlabel("Block size", fontsize=12)
ax.set_ylabel("Mean score improvement (%)", fontsize=12)
ax.set_title(f"Score improvement vs block width  (sparsity {FILTER_SPARSITY})",
             fontsize=13, fontweight="bold")
ax.grid(True, linestyle="--", alpha=0.35)
ax.legend(loc="best", frameon=False, fontsize=11)
fig.tight_layout()
plt.show()

In [ ]:
# Accuracy vs block width — companion to the score-improvement figure
# Uses the per-experiment notebooks (acc_df was loaded in section 3)

if acc_df is not None and "block_cols" not in acc_df.columns:
    # Pull block_cols out of the notebook filename if it's not in scraps
    acc_df["block_cols"] = acc_df["notebook"].str.extract(r"_(\d+)x(\d+)_").iloc[:, 1].astype(int)

acc_sweep = acc_df.dropna(subset=["acc_all"]).copy()
if len(acc_sweep) > 0:
    fig, ax = plt.subplots(figsize=(11, 6))
    for alg in sorted(acc_sweep.algorithm.unique()):
        sub = acc_sweep[acc_sweep["algorithm"] == alg].sort_values("block_cols")
        ax.plot(sub["block_cols"], sub["acc_all"], marker="o", linewidth=2.0,
                markersize=8, color=alg_color(alg), label=alg)
    if not acc_sweep["baseline_acc"].isna().all():
        baseline = acc_sweep["baseline_acc"].dropna().iloc[0]
        ax.axhline(baseline, color="black", linewidth=1.5, linestyle="--",
                   label=f"unpruned baseline ({baseline:.3f})")
    ax.set_xscale("log", base=2)
    ax.set_xticks([2, 4, 8, 16, 32])
    ax.set_xticklabels(["1×2", "1×4", "1×8", "1×16", "1×32"])
    ax.set_xlabel("Block size", fontsize=12)
    ax.set_ylabel("ImageNet val accuracy after pruning ALL layers", fontsize=12)
    ax.set_title("The accuracy cliff: ImageNet accuracy vs block width",
                 fontsize=13, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend(loc="best", frameon=False, fontsize=11)
    fig.tight_layout()
    plt.show()
else:
    print("No accuracy data — run with SKIP_ACCURACY=False first.")